In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, LSTM, Dense, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report


In [3]:
def load_dataset(file_path):
    try:
        dataset = pd.read_csv(file_path)
        print(f"Dataset loaded successfully. Shape: {dataset.shape}")
        return dataset
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return None

# Example usage
file_path = "/Users/wins/ML_Projects/LSTM_CNN_Desalination/Data/UAE_Pump_Data.csv"
dataset = load_dataset(file_path)

Dataset loaded successfully. Shape: (20000, 18)


In [25]:
def preprocess_data(dataset):
    """
    Preprocess the dataset.
    
    Parameters:
    dataset (pd.DataFrame): The dataset to preprocess.
    
    Returns:
    tuple: The preprocessed training and testing datasets.
    """
    # Drop the Timestamp column and any non-numeric columns if present
    dataset = dataset.drop(columns=['Timestamp'])

    # Separate features and target
    X = dataset.drop(columns=['Failure']).values
    y = dataset['Failure'].values

    # Normalize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Reshape data to fit the model input (samples, timesteps, features)
    # Assuming each sample is a single timestep
    X_scaled = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

    # Split the data into training and testing sets (75% training, 25% testing)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

    return X_train, X_test, y_train, y_test

# Example usage
file_path = "/Users/wins/ML_Projects/LSTM_CNN_Desalination/Data/UAE_Pump_Data.csv"
dataset = pd.read_csv(file_path)
X_train, X_test, y_train, y_test = preprocess_data(dataset)

In [26]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, LSTM, Dense, concatenate
from tensorflow.keras.models import Model

def create_parallel_lstm_cnn_model(input_shape):
    """
    Create a hybrid LSTM-CNN model.
    
    Parameters:
    input_shape (tuple): The shape of the input data.
    
    Returns:
    tf.keras.Model: The hybrid LSTM-CNN model.
    """
    # Define CNN branch
    cnn_input = Input(shape=input_shape, name='cnn_input')
    x = Conv1D(filters=32, kernel_size=3, activation='relu')(cnn_input)
    x = MaxPooling1D(pool_size=2)(x)
    x = Flatten()(x)
    cnn_output = Dense(64, activation='relu')(x)

    # Define LSTM branch
    lstm_input = Input(shape=input_shape, name='lstm_input')
    y = LSTM(64, return_sequences=True)(lstm_input)
    y = LSTM(32)(y)
    lstm_output = Dense(64, activation='relu')(y)

    # Concatenate CNN and LSTM branches
    combined = concatenate([cnn_output, lstm_output])

    # Final output layer
    output = Dense(1, activation='sigmoid')(combined)

    # Create model
    model = Model(inputs=[cnn_input, lstm_input], outputs=output)
    
    return model

# Example usage
input_shape = (X_train.shape[1], 1)  # Example input shape, adjust based on your data
model = create_parallel_lstm_cnn_model(input_shape)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cnn_input           │ (None, 16, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 14, 32)    │        128 │ cnn_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_input          │ (None, 16, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_8     │ (None, 7, 32)     │          0 │ conv1d_8[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_16 (LSTM)      │ (None, 16, 64)    │     16,896 │ lstm_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_8 (Flatten) │ (None, 224)       │          0 │ max_pooling1d_8[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_17 (LSTM)      │ (None, 32)        │     12,416 │ lstm_16[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 64)        │     14,400 │ flatten_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 64)        │      2,112 │ lstm_17[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 128)       │          0 │ dense_24[0][0],   │
│ (Concatenate)       │                   │            │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 1)         │        129 │ concatenate_8[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 46,081 (180.00 KB)

 Trainable params: 46,081 (180.00 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
# Example usage
input_shape = (X_train.shape[1], 1)
model = create_parallel_lstm_cnn_model(input_shape)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(
    [X_train, X_train], y_train,  # using the same data for both inputs for simplicity
    epochs=100,
    batch_size=32,
    validation_data=([X_test, X_test], y_test),
    callbacks=[early_stopping]
)

# Save the model in the native Keras format
model.save('hybrid_lstm_cnn_model.keras')

Epoch 1/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9800 - loss: 0.0810 - val_accuracy: 1.0000 - val_loss: 5.2290e-05
Epoch 2/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 1.0000 - loss: 2.4780e-05 - val_accuracy: 1.0000 - val_loss: 1.1712e-05
Epoch 3/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 1.0000 - loss: 5.0169e-06 - val_accuracy: 1.0000 - val_loss: 3.9185e-06
Epoch 4/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 1.9497e-06 - val_accuracy: 1.0000 - val_loss: 2.0102e-06
Epoch 5/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 1.0000 - loss: 1.0611e-06 - val_accuracy: 1.0000 - val_loss: 1.1732e-06
Epoch 6/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 5.1672e-07 - val_accuracy: 1.0000 - val_loss: 7.2963e-07
Epoch 7/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 3.5149e-07 - val_accuracy: 1.0000 - val_loss: 4.8314e-07
Epoch 8/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5

In [28]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

def load_model(model_path):
    """
    Load the trained model from the specified path.
    
    Parameters:
    model_path (str): The path to the trained model file.
    
    Returns:
    tf.keras.Model: The loaded model.
    """
    return tf.keras.models.load_model(model_path)

def evaluate_model(model, X_test, y_test):
    """
    Evaluate the model on the test data.
    
    Parameters:
    model (tf.keras.Model): The trained model.
    X_test (np.array): The test features.
    y_test (np.array): The test labels.
    
    Returns:
    None
    """
    # Make predictions
    y_pred_proba = model.predict([X_test, X_test])
    y_pred = (y_pred_proba > 0.5).astype("int32")

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)

    # Print evaluation metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)
    print("Classification Report:")
    print(class_report)

# Example usage
model_path = '/Users/wins/ML_Projects/LSTM_CNN_Desalination/hybrid_lstm_cnn_model.keras'
model = load_model(model_path)

# Assuming X_test and y_test are already defined from the previous preprocessing step
evaluate_model(model, X_test, y_test)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
Confusion Matrix:
[[3538    0]
 [   0 1462]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3538
           1       1.00      1.00      1.00      1462

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

